# DNAK — Reproducing the paper's claims (Google Colab)

**Paper:** *DNAK: Exact Acceleration of Lloyd's k-Means in Linear Memory via
Delaunay-Neighbor Pruning* (B. Kr. Paul, K. Giri).
Code: https://github.com/barunkrpaul/dnak-kmeans

Run the cells **in order, top to bottom** (Runtime → Run all). This notebook checks:

1. **Exactness** — every method returns a clustering identical to Lloyd's (asserted).
2. **Six-method benchmark** — wall-clock and distance-computation comparison.
3. **The crossover** — DNAK vs Exponion as k grows.

Colab CPUs vary between sessions, so absolute times differ from the paper; what
this notebook checks is the *ordering* of the methods and the crossover trend.
Problem sizes are reduced so everything finishes in roughly 10–15 minutes.

In [ ]:
%pip -q install numba
import numba, numpy, scipy, sklearn
print('numba', numba.__version__, '| numpy', numpy.__version__, '| scipy', scipy.__version__)

## Algorithm definitions\nRun this cell before any cell below it. It defines all six algorithms and `run_algo`.

In [ ]:
"""
DNAK: Delaunay-Neighbor Accelerated K-means (exact acceleration of Lloyd's).
Baselines: Lloyd's algorithm (standard) and Hamerly's algorithm (bound-based accelerator).
Primary metric: number of point-to-centroid distance computations (implementation-independent).
Author: prototype for Barun Kr Paul's PhD research.
"""
import numpy as np
import time
import csv
from scipy.spatial import Delaunay
try:
    from scipy.spatial import QhullError
except ImportError:
    from scipy.spatial.qhull import QhullError

rng_global = np.random.default_rng(42)

# ---------------------------------------------------------------- data / init

def make_blobs(n, d, k_true, seed, spread=0.6, box=10.0):
    rng = np.random.default_rng(seed)
    centers = rng.uniform(-box, box, size=(k_true, d))
    sizes = rng.multinomial(n, np.ones(k_true) / k_true)
    parts = [rng.normal(c, spread, size=(s, d)) for c, s in zip(centers, sizes)]
    X = np.vstack(parts)
    rng.shuffle(X)
    return X.astype(np.float64)

def kmeanspp_init(X, k, seed):
    """Standard k-means++ seeding (shared identically by all algorithms)."""
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    C = np.empty((k, X.shape[1]))
    C[0] = X[rng.integers(n)]
    d2 = np.sum((X - C[0]) ** 2, axis=1)
    for j in range(1, k):
        probs = d2 / d2.sum()
        C[j] = X[rng.choice(n, p=probs)]
        d2 = np.minimum(d2, np.sum((X - C[j]) ** 2, axis=1))
    return C

def sse_of(X, labels, C):
    return float(np.sum((X - C[labels]) ** 2))

def update_centroids(X, labels, C_old, k):
    C = C_old.copy()
    for j in range(k):
        pts = X[labels == j]
        if len(pts) > 0:
            C[j] = pts.mean(axis=0)
    return C

# ---------------------------------------------------------------- Lloyd's

def lloyd(X, C0, max_iter=300):
    n, d = X.shape
    k = C0.shape[0]
    C = C0.copy()
    labels = np.full(n, -1)
    comps = 0
    t0 = time.perf_counter()
    for it in range(1, max_iter + 1):
        D = np.linalg.norm(X[:, None, :] - C[None, :, :], axis=2)  # n x k
        comps += n * k
        new_labels = np.argmin(D, axis=1)
        if it > 1:  # tie rule: keep current assignment unless strictly closer
            dmin = D[np.arange(n), new_labels]
            keep = D[np.arange(n), labels] <= dmin
            new_labels[keep] = labels[keep]
        changed = int(np.sum(new_labels != labels))
        labels = new_labels
        C = update_centroids(X, labels, C, k)
        if changed == 0:
            break
    t = time.perf_counter() - t0
    return dict(name="Lloyd", labels=labels, C=C, iters=it,
                comps=comps, time=t, sse=sse_of(X, labels, C))

# ---------------------------------------------------------------- Hamerly

def hamerly(X, C0, max_iter=300):
    n, d = X.shape
    k = C0.shape[0]
    C = C0.copy()
    comps = 0
    cc_comps = 0  # centroid-centroid distances (overhead, O(k^2) per iter)
    t0 = time.perf_counter()

    # initial full scan
    D = np.linalg.norm(X[:, None, :] - C[None, :, :], axis=2)
    comps += n * k
    order = np.argsort(D, axis=1)
    a = order[:, 0]
    u = D[np.arange(n), a]
    l = D[np.arange(n), order[:, 1]]

    for it in range(1, max_iter + 1):
        Cn = update_centroids(X, a, C, k)
        drift = np.linalg.norm(Cn - C, axis=1)
        C = Cn
        dmax = drift.max()
        # s(j) = 0.5 * dist to nearest other centroid
        CC = np.linalg.norm(C[:, None, :] - C[None, :, :], axis=2)
        cc_comps += k * k
        np.fill_diagonal(CC, np.inf)
        s = 0.5 * CC.min(axis=1)

        u = u + drift[a]
        l = l - dmax

        m = np.maximum(s[a], l)
        cand = np.where(u > m)[0]
        # tighten u for candidates
        if len(cand) > 0:
            u[cand] = np.linalg.norm(X[cand] - C[a[cand]], axis=1)
            comps += len(cand)
            cand2 = cand[u[cand] > m[cand]]
        else:
            cand2 = cand
        changed = 0
        if len(cand2) > 0:
            D2 = np.linalg.norm(X[cand2, None, :] - C[None, :, :], axis=2)
            comps += len(cand2) * k
            order2 = np.argsort(D2, axis=1)
            na = order2[:, 0]
            dmin2 = D2[np.arange(len(cand2)), na]
            keep = D2[np.arange(len(cand2)), a[cand2]] <= dmin2
            na[keep] = a[cand2][keep]  # tie rule: keep current
            changed = int(np.sum(na != a[cand2]))
            a[cand2] = na
            u[cand2] = D2[np.arange(len(cand2)), na]
            # lower bound: min distance among centroids != assigned
            D2m = D2.copy()
            D2m[np.arange(len(cand2)), na] = np.inf
            l[cand2] = D2m.min(axis=1)
        if changed == 0 and dmax == 0.0:
            break
        if changed == 0:
            # one more centroid update pass to confirm convergence
            Ct = update_centroids(X, a, C, k)
            if np.allclose(Ct, C):
                break
    t = time.perf_counter() - t0
    return dict(name="Hamerly", labels=a, C=C, iters=it,
                comps=comps, cc_comps=cc_comps, time=t, sse=sse_of(X, a, C))

# ---------------------------------------------------------------- DNAK

def delaunay_adjacency(C):
    """Adjacency lists of the Delaunay graph over centroids."""
    k = C.shape[0]
    try:
        tri = Delaunay(C)
    except QhullError:
        tri = Delaunay(C, qhull_options="QJ")
    adj = [set() for _ in range(k)]
    for simplex in tri.simplices:
        for i in range(len(simplex)):
            for j in range(i + 1, len(simplex)):
                adj[simplex[i]].add(simplex[j])
                adj[simplex[j]].add(simplex[i])
    return [np.fromiter(s, dtype=np.int64) for s in adj]

def dnak(X, C0, max_iter=300):
    n, d = X.shape
    k = C0.shape[0]
    C = C0.copy()
    comps = 0
    graph_time = 0.0
    skipped_total = 0
    walk_hops_total = 0
    t0 = time.perf_counter()

    # initial full scan (exact u and l)
    D = np.linalg.norm(X[:, None, :] - C[None, :, :], axis=2)
    comps += n * k
    order = np.argsort(D, axis=1)
    a = order[:, 0]
    u = D[np.arange(n), a]
    l = D[np.arange(n), order[:, 1]]

    for it in range(1, max_iter + 1):
        Cn = update_centroids(X, a, C, k)
        drift = np.linalg.norm(Cn - C, axis=1)
        C = Cn
        dmax = drift.max()

        tg = time.perf_counter()
        adj = delaunay_adjacency(C)
        graph_time += time.perf_counter() - tg

        u = u + drift[a]
        l = l - dmax

        changed = 0
        for i in range(n):
            if u[i] <= l[i]:
                skipped_total += 1
                continue
            # tighten upper bound
            du = np.linalg.norm(X[i] - C[a[i]])
            comps += 1
            u[i] = du
            if u[i] <= l[i]:
                skipped_total += 1
                continue
            # Delaunay greedy walk
            cur = a[i]
            cur_d = du
            hops = 0
            while True:
                nb = adj[cur]
                nd = np.linalg.norm(X[i] - C[nb], axis=1)
                comps += len(nb)
                jmin = int(np.argmin(nd))
                if nd[jmin] < cur_d:
                    cur = int(nb[jmin])
                    cur_d = float(nd[jmin])
                    hops += 1
                else:
                    # local min on Delaunay graph = global nearest (exact)
                    second = float(nd[jmin])  # min over neighbors of final site
                    break
            walk_hops_total += hops
            if cur != a[i]:
                changed += 1
                a[i] = cur
            u[i] = cur_d
            l[i] = second  # exact 2nd-nearest via order-2 Voronoi lemma
        if changed == 0 and dmax < 1e-12:
            break
        if changed == 0:
            Ct = update_centroids(X, a, C, k)
            if np.allclose(Ct, C):
                break
    t = time.perf_counter() - t0
    return dict(name="DNAK", labels=a, C=C, iters=it, comps=comps,
                time=t, graph_time=graph_time, sse=sse_of(X, a, C),
                skipped=skipped_total, hops=walk_hops_total)

# ---------------------------------------------------------------- benchmark

def run_config(n, d, k, seed):
    X = make_blobs(n, d, k_true=k, seed=seed)
    C0 = kmeanspp_init(X, k, seed=seed + 1)
    rL = lloyd(X, C0)
    rH = hamerly(X, C0)
    rD = dnak(X, C0)

    exact_H = np.array_equal(rL["labels"], rH["labels"])
    exact_D = np.array_equal(rL["labels"], rD["labels"])
    sse_ok = (abs(rL["sse"] - rD["sse"]) / rL["sse"] < 1e-10 and
              abs(rL["sse"] - rH["sse"]) / rL["sse"] < 1e-10)

    rows = []
    for r in (rL, rH, rD):
        rows.append(dict(n=n, d=d, k=k, algo=r["name"], iters=r["iters"],
                         comps=r["comps"], time=round(r["time"], 3),
                         sse=round(r["sse"], 2),
                         reduction_vs_lloyd=round(rL["comps"] / r["comps"], 2)))
    print(f"\n=== n={n} d={d} k={k} ===")
    print(f"identical labels: Hamerly={exact_H}  DNAK={exact_D}  SSE match={sse_ok}")
    for row in rows:
        print(f"  {row['algo']:8s} iters={row['iters']:3d} "
              f"dist_comps={row['comps']:>12,d}  "
              f"reduction={row['reduction_vs_lloyd']:6.2f}x  "
              f"time={row['time']:7.3f}s  SSE={row['sse']:.2f}")
    if not (exact_D and sse_ok):
        print("  !!! EXACTNESS VIOLATION — DO NOT USE THESE RESULTS !!!")
    return rows, exact_D and sse_ok

"""
Compiled (Numba) implementations of Lloyd, Hamerly, Elkan and DNAK,
with a wall-clock benchmark. All algorithms share the tie rule
"switch only if strictly closer"; exactness (identical SSE) asserted.
JIT compilation is excluded from timing via a warm-up pass.
"""
import numpy as np, time, csv
from numba import njit
from scipy.spatial import Delaunay

# ------------------------------------------------------------ kernels
@njit
def _dist(X, i, C, j):
    s = 0.0
    for t in range(X.shape[1]):
        dv = X[i, t] - C[j, t]
        s += dv * dv
    return np.sqrt(s)

@njit
def full_scan_init(X, C, a, u, l):
    n, k = X.shape[0], C.shape[0]
    comps = 0
    for i in range(n):
        b1 = 1e300; b2 = 1e300; bj = -1
        for j in range(k):
            d = _dist(X, i, C, j); comps += 1
            if d < b1:
                b2 = b1; b1 = d; bj = j
            elif d < b2:
                b2 = d
        a[i] = bj; u[i] = b1; l[i] = b2
    return comps

@njit
def centroid_update(X, a, C_old, k):
    n, d = X.shape
    C = np.zeros((k, d)); cnt = np.zeros(k)
    for i in range(n):
        cnt[a[i]] += 1.0
        for t in range(d):
            C[a[i], t] += X[i, t]
    for j in range(k):
        if cnt[j] > 0:
            for t in range(d):
                C[j, t] /= cnt[j]
        else:
            for t in range(d):
                C[j, t] = C_old[j, t]
    return C

@njit
def lloyd_iter(X, C, a):
    n, k = X.shape[0], C.shape[0]
    changed = 0; comps = 0
    for i in range(n):
        cur = a[i]
        best = _dist(X, i, C, cur); bj = cur; comps += 1
        for j in range(k):
            if j == cur:
                continue
            d = _dist(X, i, C, j); comps += 1
            if d < best:            # strictly closer only (tie rule)
                best = d; bj = j
        if bj != cur:
            a[i] = bj; changed += 1
    return changed, comps

@njit
def hamerly_iter(X, C, a, u, l, drift, dmax, s):
    n, k = X.shape[0], C.shape[0]
    changed = 0; comps = 0
    for i in range(n):
        u[i] += drift[a[i]]; l[i] -= dmax
        m = s[a[i]] if s[a[i]] > l[i] else l[i]
        if u[i] <= m:
            continue
        u[i] = _dist(X, i, C, a[i]); comps += 1
        if u[i] <= m:
            continue
        cur = a[i]
        b1 = u[i]; bj = cur; b2 = 1e300
        for j in range(k):
            if j == cur:
                continue
            d = _dist(X, i, C, j); comps += 1
            if d < b1:              # strictly closer (tie rule)
                b2 = b1; b1 = d; bj = j
            elif d < b2:
                b2 = d
        if bj != cur:
            a[i] = bj; changed += 1
        u[i] = b1; l[i] = b2
    return changed, comps

@njit
def elkan_iter(X, C, a, u, L, CC, s, drift):
    n, k = X.shape[0], C.shape[0]
    changed = 0; comps = 0
    for i in range(n):
        u[i] += drift[a[i]]
        for j in range(k):
            L[i, j] -= drift[j]
            if L[i, j] < 0.0:
                L[i, j] = 0.0
        if u[i] <= s[a[i]]:
            continue
        tight = False
        old = a[i]
        for j in range(k):
            if j == a[i]:
                continue
            if u[i] > L[i, j] and u[i] > 0.5 * CC[a[i], j]:
                if not tight:
                    u[i] = _dist(X, i, C, a[i]); comps += 1
                    L[i, a[i]] = u[i]; tight = True
                    if u[i] <= L[i, j] or u[i] <= 0.5 * CC[a[i], j]:
                        continue
                d = _dist(X, i, C, j); comps += 1
                L[i, j] = d
                if d < u[i]:        # strictly closer (tie rule)
                    a[i] = j; u[i] = d
        if a[i] != old:
            changed += 1
    return changed, comps

@njit
def dnak_iter(X, C, a, u, l, drift, dmax, indptr, indices):
    n = X.shape[0]
    changed = 0; comps = 0
    for i in range(n):
        u[i] += drift[a[i]]; l[i] -= dmax
        if u[i] <= l[i]:
            continue
        u[i] = _dist(X, i, C, a[i]); comps += 1
        if u[i] <= l[i]:
            continue
        cur = a[i]; curd = u[i]; second = 1e300
        while True:
            b = 1e300; bj = -1
            for p in range(indptr[cur], indptr[cur + 1]):
                j = indices[p]
                d = _dist(X, i, C, j); comps += 1
                if d < b:
                    b = d; bj = j
            if b < curd:            # strictly closer: hop
                cur = bj; curd = b
            else:                   # local min = global nearest
                second = b
                break
        if cur != a[i]:
            a[i] = cur; changed += 1
        u[i] = curd; l[i] = second
    return changed, comps

# ------------------------------------------------------------ drivers
def _cc_s(C):
    CC = np.sqrt(((C[:, None, :] - C[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(CC, np.inf)
    return CC, 0.5 * CC.min(axis=1)

def _adjacency_csr(C):
    try:
        tri = Delaunay(C)
    except Exception:
        tri = Delaunay(C, qhull_options="QJ")
    k = C.shape[0]
    adj = [set() for _ in range(k)]
    for sx in tri.simplices:
        for ii in range(len(sx)):
            for jj in range(ii + 1, len(sx)):
                adj[sx[ii]].add(sx[jj]); adj[sx[jj]].add(sx[ii])
    indptr = np.zeros(k + 1, np.int64)
    for j in range(k):
        indptr[j + 1] = indptr[j] + len(adj[j])
    indices = np.empty(indptr[-1], np.int64)
    for j in range(k):
        indices[indptr[j]:indptr[j + 1]] = sorted(adj[j])
    return indptr, indices

def _sse(X, a, C):
    return float(((X - C[a]) ** 2).sum())

def run_algo(name, X, C0, max_iter=300):
    n, d = X.shape; k = C0.shape[0]
    C = C0.copy()
    a = np.zeros(n, np.int64); u = np.zeros(n); l = np.zeros(n)
    t0 = time.perf_counter()
    comps = full_scan_init(X, C, a, u, l)
    L = None
    if name == "Elkan":
        L = np.sqrt(((X[:, None, :] - C[None, :, :]) ** 2).sum(-1))
    for it in range(1, max_iter + 1):
        Cn = centroid_update(X, a, C, k)
        drift = np.sqrt(((Cn - C) ** 2).sum(-1)); dmax = drift.max()
        C = Cn
        if name == "Lloyd":
            ch, cp = lloyd_iter(X, C, a)
        elif name == "Hamerly":
            CC, s = _cc_s(C)
            ch, cp = hamerly_iter(X, C, a, u, l, drift, dmax, s)
        elif name == "Elkan":
            CC, s = _cc_s(C)
            ch, cp = elkan_iter(X, C, a, u, L, CC, s, drift)
        else:  # DNAK
            indptr, indices = _adjacency_csr(C)
            ch, cp = dnak_iter(X, C, a, u, l, drift, dmax, indptr, indices)
        comps += cp
        if ch == 0:
            Ct = centroid_update(X, a, C, k)
            if np.allclose(Ct, C):
                break
    t = time.perf_counter() - t0
    return dict(name=name, labels=a, C=C, iters=it, comps=comps,
                time=t, sse=_sse(X, a, C))

def warmup():
    X = make_blobs(500, 2, 4, seed=0)
    C0 = kmeanspp_init(X, 4, seed=0)
    for nm in ("Lloyd", "Hamerly", "Elkan", "DNAK"):
        run_algo(nm, X, C0, max_iter=5)

def load_full_image(name):
    from sklearn.datasets import load_sample_image
    rng = np.random.default_rng(0)
    P = load_sample_image(name).astype(np.float64).reshape(-1, 3) / 255.0
    return P + rng.uniform(-1e-7, 1e-7, size=P.shape)

"""
Benchmark v3 (compiled): Lloyd, Hamerly, Elkan, Yinyang, Exponion, DNAK.
10 initialisation seeds per configuration. Reports BOTH distance
computations and wall-clock from the same Numba-compiled runs.
Exactness (identical SSE to Lloyd, shared strict tie rule) asserted on
every run. Yinyang = global + group filter (Ding et al. 2015);
Exponion = annular search over centroid-centroid sorted lists
(Newling & Fleuret 2016). Comparison metric of record is distance
computations, which is implementation-independent.
"""
import sys, time, csv
import numpy as np
from numba import njit

# ---------------------------------------------------------------- Yinyang
@njit
def yy_init(X, C, g_of, t, a, u, LB):
    n, k = X.shape[0], C.shape[0]
    comps = 0
    for i in range(n):
        b1 = 1e300; bj = -1
        for j in range(k):
            d = _dist(X, i, C, j); comps += 1
            if d < b1: b1 = d; bj = j
        a[i] = bj; u[i] = b1
        for gg in range(t): LB[i, gg] = 1e300
        for j in range(k):
            if j == bj: continue
            d = _dist(X, i, C, j); comps += 1
            if d < LB[i, g_of[j]]: LB[i, g_of[j]] = d
    return comps

@njit
def yy_iter(X, C, a, u, LB, drift, gd, g_of, gptr, gidx, t):
    n = X.shape[0]
    changed = 0; comps = 0
    gm1 = np.empty(t); gm2 = np.empty(t)
    gm1j = np.empty(t, np.int64); scanned = np.empty(t, np.bool_)
    for i in range(n):
        u[i] += drift[a[i]]
        gmin = 1e300
        for gg in range(t):
            LB[i, gg] -= gd[gg]
            if LB[i, gg] < gmin: gmin = LB[i, gg]
        if u[i] <= gmin: continue
        old = a[i]
        ut = _dist(X, i, C, old); comps += 1
        u[i] = ut
        if u[i] <= gmin: continue
        b1 = ut; b1j = old
        for gg in range(t):
            scanned[gg] = False
            if ut <= LB[i, gg] and gg != g_of[old]:
                continue  # group passes filter (old a's group re-scanned)
            scanned[gg] = True
            m1 = 1e300; m1j = -1; m2 = 1e300
            if gg == g_of[old]:
                m1 = ut; m1j = old
            for p in range(gptr[gg], gptr[gg + 1]):
                j = gidx[p]
                if j == old: continue
                d = _dist(X, i, C, j); comps += 1
                if d < m1: m2 = m1; m1 = d; m1j = j
                elif d < m2: m2 = d
                if d < b1: b1 = d; b1j = j
            gm1[gg] = m1; gm2[gg] = m2; gm1j[gg] = m1j
        # finalise bounds only now, against the FINAL winner b1j
        for gg in range(t):
            if not scanned[gg]: continue
            LB[i, gg] = gm2[gg] if gm1j[gg] == b1j else gm1[gg]
        if b1j != old:
            go = g_of[old]  # old centroid rejoins its group's bound
            if ut < LB[i, go]: LB[i, go] = ut
            a[i] = b1j; changed += 1
        u[i] = b1
    return changed, comps

def make_groups(C0, t, seed=0):
    """Static centroid grouping by a short k-means on the centroids."""
    rng = np.random.default_rng(seed)
    k = C0.shape[0]
    G = C0[rng.choice(k, size=t, replace=False)].copy()
    for _ in range(8):
        lab = np.argmin(((C0[:, None, :] - G[None, :, :]) ** 2).sum(-1), 1)
        for gg in range(t):
            if np.any(lab == gg): G[gg] = C0[lab == gg].mean(0)
    lab = np.argmin(((C0[:, None, :] - G[None, :, :]) ** 2).sum(-1), 1)
    order = np.argsort(lab, kind="stable")
    gptr = np.zeros(t + 1, np.int64)
    for gg in range(t): gptr[gg + 1] = gptr[gg] + int((lab == gg).sum())
    return lab.astype(np.int64), gptr, order.astype(np.int64)

# ---------------------------------------------------------------- Exponion
@njit
def exp_iter(X, C, a, u, l, drift, dmax, sidx, sval):
    n = X.shape[0]; km1 = sidx.shape[1]
    changed = 0; comps = 0
    for i in range(n):
        u[i] += drift[a[i]]; l[i] -= dmax
        if u[i] <= l[i]: continue
        old = a[i]
        ut = _dist(X, i, C, old); comps += 1
        u[i] = ut
        if u[i] <= l[i]: continue
        r = 2.0 * ut
        b1 = ut; b1j = old; b2 = 1e300
        p = 0
        while p < km1 and sval[old, p] <= r:
            j = sidx[old, p]
            d = _dist(X, i, C, j); comps += 1
            if d < b1: b2 = b1; b1 = d; b1j = j
            elif d < b2: b2 = d
            p += 1
        outer = (sval[old, p] - ut) if p < km1 else 1e300
        if b1j != old:
            a[i] = b1j; changed += 1
        u[i] = b1
        l[i] = b2 if b2 < outer else outer
    return changed, comps

# ---------------------------------------------------------------- driver
def run_algo(name, X, C0, max_iter=300):
    n, d = X.shape; k = C0.shape[0]
    C = C0.copy()
    a = np.zeros(n, np.int64); u = np.zeros(n); l = np.zeros(n)
    t0 = time.perf_counter()
    if name == "Yinyang":
        t = max(1, k // 10)
        g_of, gptr, gidx = make_groups(C0, t)
        LB = np.zeros((n, t))
        comps = yy_init(X, C, g_of, t, a, u, LB)
    else:
        comps = full_scan_init(X, C, a, u, l)
        L = None
        if name == "Elkan":
            L = np.sqrt(((X[:, None, :] - C[None, :, :]) ** 2).sum(-1))
    for it in range(1, max_iter + 1):
        Cn = centroid_update(X, a, C, k)
        drift = np.sqrt(((Cn - C) ** 2).sum(-1)); dmax = drift.max()
        C = Cn
        if name == "Lloyd":
            ch, cp = lloyd_iter(X, C, a)
        elif name == "Hamerly":
            CC, s = _cc_s(C)
            ch, cp = hamerly_iter(X, C, a, u, l, drift, dmax, s)
        elif name == "Elkan":
            CC, s = _cc_s(C)
            ch, cp = elkan_iter(X, C, a, u, L, CC, s, drift)
        elif name == "Yinyang":
            gd = np.zeros(len(gptr) - 1)
            for gg in range(len(gd)):
                if gptr[gg + 1] > gptr[gg]:
                    gd[gg] = drift[gidx[gptr[gg]:gptr[gg + 1]]].max()
            ch, cp = yy_iter(X, C, a, u, LB, drift, gd, g_of, gptr, gidx,
                             len(gd))
        elif name == "Exponion":
            CC = np.sqrt(((C[:, None, :] - C[None, :, :]) ** 2).sum(-1))
            np.fill_diagonal(CC, np.inf)
            sidx = np.argsort(CC, axis=1)[:, :k - 1].astype(np.int64)
            sval = np.take_along_axis(CC, sidx, axis=1)
            ch, cp = exp_iter(X, C, a, u, l, drift, dmax, sidx, sval)
        else:  # DNAK
            indptr, indices = _adjacency_csr(C)
            ch, cp = dnak_iter(X, C, a, u, l, drift, dmax, indptr, indices)
        comps += cp
        if ch == 0:
            Ct = centroid_update(X, a, C, k)
            if np.allclose(Ct, C): break
    tm = time.perf_counter() - t0
    return dict(name=name, labels=a, iters=it, comps=comps, time=tm,
                sse=_sse(X, a, C))

ALGOS = ["Lloyd", "Hamerly", "Elkan", "Yinyang", "Exponion", "DNAK"]

def warmup():
    X = make_blobs(600, 2, 4, seed=0)
    C0 = kmeanspp_init(X, 4, seed=0)
    rL = run_algo("Lloyd", X, C0, 8)
    for nm in ALGOS[1:]:
        run_algo(nm, X, C0, 8)

def verify():
    ok = True
    for seed in (1, 2, 3):
        X = make_blobs(4000, 3, 24, seed=seed)
        C0 = kmeanspp_init(X, 24, seed=seed + 50)
        rL = run_algo("Lloyd", X, C0)
        for nm in ALGOS[1:]:
            r = run_algo(nm, X, C0)
            e = abs(r["sse"] - rL["sse"]) / rL["sse"]
            if e > 1e-10:
                print("VERIFY FAIL", nm, "seed", seed, "rel err", e); ok = False
    print("verification:", "ALL SIX ALGORITHMS EXACT" if ok else "FAILED")
    return ok

def bench(configs, outfile, seeds):
    rows = []
    for cname, X, k in configs:
        agg_c = {nm: [] for nm in ALGOS}; agg_t = {nm: [] for nm in ALGOS}
        for seed in seeds:
            C0 = kmeanspp_init(X, k, seed)
            rL = run_algo("Lloyd", X, C0)
            agg_c["Lloyd"].append(rL["comps"]); agg_t["Lloyd"].append(rL["time"])
            for nm in ALGOS[1:]:
                r = run_algo(nm, X, C0)
                assert abs(r["sse"] - rL["sse"]) / rL["sse"] < 1e-10, \
                    f"SSE mismatch {nm} {cname} seed {seed}"
                agg_c[nm].append(r["comps"]); agg_t[nm].append(r["time"])
        print(f"\n=== {cname} ({len(seeds)} seeds, all exact) ===")
        for nm in ALGOS:
            c = np.array(agg_c[nm], float); tt = np.array(agg_t[nm], float)
            rc = np.array(agg_c["Lloyd"]) / c
            rt = np.array(agg_t["Lloyd"]) / tt
            print(f"  {nm:9s} DCred={rc.mean():6.2f}x±{rc.std():5.2f}"
                  f"  speedup={rt.mean():5.2f}x±{rt.std():4.2f}"
                  f"  time={tt.mean():6.2f}s")
            rows.append(dict(config=cname, algo=nm,
                comps_mean=int(c.mean()), dcred_mean=round(rc.mean(), 2),
                dcred_std=round(rc.std(), 2), time_mean=round(tt.mean(), 3),
                sp_mean=round(rt.mean(), 2), sp_std=round(rt.std(), 2)))
    mode = "a" if outfile == "APPEND" else "w"
    path = "/home/claude/results_v3.csv"
    import os
    write_header = not (mode == "a" and os.path.exists(path))
    with open(path, mode if mode == "a" else "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        if write_header: w.writeheader()
        w.writerows(rows)



In [ ]:
# Sanity check: confirm the definitions cell above ran successfully.
for _f in ("run_algo", "make_blobs", "kmeanspp_init", "dnak_iter", "exp_iter", "yy_iter"):
    assert _f in globals(), f"{_f} missing - run the cell above (Runtime > Run all)"
print("OK: all algorithm definitions loaded")

In [ ]:
# Claim 1: EXACTNESS — every algorithm must match Lloyd's SSE exactly
import numpy as np
ALGOS = ["Lloyd", "Hamerly", "Elkan", "Yinyang", "Exponion", "DNAK"]

print("compiling kernels on a small warm-up problem (takes ~1 min the first time) ...")
Xw = make_blobs(600, 2, 4, seed=0); C0w = kmeanspp_init(Xw, 4, 0)
for nm in ALGOS: run_algo(nm, Xw, C0w, 8)

print("verifying exactness on 3 seeds x 5 baselines ...")
for seed in (1, 2, 3):
    X = make_blobs(4000, 3, 24, seed=seed)
    C0 = kmeanspp_init(X, 24, seed=seed + 50)
    rL = run_algo("Lloyd", X, C0)
    for nm in ALGOS[1:]:
        r = run_algo(nm, X, C0)
        rel = abs(r["sse"] - rL["sse"]) / rL["sse"]
        assert rel < 1e-10, f"{nm} mismatch (seed {seed})"
print("PASS: all six algorithms produce identical clusterings (SSE match < 1e-10)")

In [ ]:
# Claim 2: six-method benchmark (reduced size for Colab)
import numpy as np
N_BENCH, K_BENCH, SEEDS = 50000, 64, (11, 22, 33)
X = make_blobs(N_BENCH, 2, K_BENCH, seed=21)
agg_t, agg_c = {}, {}
for seed in SEEDS:
    C0 = kmeanspp_init(X, K_BENCH, seed)
    rL = run_algo("Lloyd", X, C0)
    for nm in ALGOS:
        r = rL if nm == "Lloyd" else run_algo(nm, X, C0)
        if nm != "Lloyd":
            assert abs(r["sse"] - rL["sse"]) / rL["sse"] < 1e-10, nm
        agg_t.setdefault(nm, []).append(r["time"])
        agg_c.setdefault(nm, []).append(r["comps"])
print(f"n={N_BENCH}, d=2, k={K_BENCH}, {len(SEEDS)} seeds - all runs exact\n")
print(f"{'method':10s} {'time (s)':>9s} {'speedup':>9s} {'DC reduction':>13s}")
for nm in ALGOS:
    t = np.mean(agg_t[nm]); sp = np.mean(np.array(agg_t["Lloyd"]) / agg_t[nm])
    dc = np.mean(np.array(agg_c["Lloyd"]) / np.array(agg_c[nm], float))
    print(f"{nm:10s} {t:9.2f} {sp:8.1f}x {dc:12.1f}x")

In [ ]:
# Claim 3: the crossover — DNAK vs Exponion as k grows (d=2)
import numpy as np, matplotlib.pyplot as plt
KS, SEEDS = [64, 128, 256], (11, 22)
res = {"Exponion": [], "DNAK": []}
for k in KS:
    X = make_blobs(50000, 2, k, seed=40 + k)
    sp = {"Exponion": [], "DNAK": []}
    for seed in SEEDS:
        C0 = kmeanspp_init(X, k, seed)
        rL = run_algo("Lloyd", X, C0)
        for nm in ("Exponion", "DNAK"):
            r = run_algo(nm, X, C0)
            assert abs(r["sse"] - rL["sse"]) / rL["sse"] < 1e-10
            sp[nm].append(rL["time"] / r["time"])
    for nm in ("Exponion", "DNAK"): res[nm].append(np.mean(sp[nm]))
    print(f"k={k:3d}:  Exponion {res['Exponion'][-1]:5.1f}x   DNAK {res['DNAK'][-1]:5.1f}x")
plt.figure(figsize=(5.5, 3.2), dpi=120)
plt.plot(KS, res["Exponion"], "o--", label="Exponion")
plt.plot(KS, res["DNAK"], "s-", label="DNAK")
plt.xlabel("k"); plt.ylabel("speedup vs Lloyd"); plt.xscale("log", base=2)
plt.xticks(KS, KS); plt.legend(frameon=False); plt.title("d=2 crossover (n=50k)")
plt.tight_layout(); plt.show()
print("\nExpected: DNAK overtakes Exponion by k=128-256, matching the paper's trend.")

## Running the paper's full configurations

The paper uses n=100,000 synthetic mixtures (k up to 512) and the full
273,280-pixel image from `sklearn.datasets.load_sample_image`. To reproduce those,
raise `N_BENCH` to 100000 and extend `KS` to `[64, 96, 128, 192, 256, 384, 512]`
above — expect 1–2 hours on a Colab CPU. The full scripts and result files are in
the repository under `experiments/` and `results/`.